# RAG with Local Documents using Claude + FAISS

**Retrieval-Augmented Generation (RAG)** lets you ground a language model's answers in your own documents instead of relying solely on what it learned during training.

## What is RAG?

RAG has three phases:
1. **Index** — convert your documents into dense vector embeddings and store them in a vector index.
2. **Retrieve** — embed the user's question and find the most similar document chunks.
3. **Generate** — pass those chunks as context to Claude and ask it to answer.

## When to use RAG

- Your information changes frequently (docs, wikis, changelogs).
- Your data is private and you can't fine-tune a public model on it.
- You need citations — RAG naturally tells you *where* the answer came from.
- The knowledge base is too large to fit in a single context window.

This notebook walks through every step from scratch using:
- **`sentence-transformers`** for local, free embeddings
- **FAISS** for fast nearest-neighbour search
- **Claude** (`claude-opus-4-7`) for final answer generation

No LangChain or LlamaIndex — just plain Python so you can see exactly what's happening.

## Install dependencies

In [ ]:
%pip install anthropic faiss-cpu sentence-transformers pdfplumber

## Setup

Import the libraries we need and initialise the Anthropic client.
Set `ANTHROPIC_API_KEY` in your environment before running (e.g. `export ANTHROPIC_API_KEY=sk-ant-...`).

In [ ]:
import os
import textwrap
from pathlib import Path

import anthropic
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("Anthropic client ready.")

## Step 1 — Load and chunk documents

We read every `.md` and `.txt` file in a directory and split each file into overlapping chunks.
Overlapping chunks (64-char overlap on 512-char windows) prevent important context from being cut at a boundary.

In [ ]:
def load_and_chunk(directory: str, chunk_size: int = 512, overlap: int = 64) -> list[dict]:
    """
    Walk *directory* and split every .md / .txt file into overlapping chunks.

    Returns a list of dicts, each with:
      - 'text'   : the chunk content
      - 'source' : the originating file path
      - 'chunk'  : zero-based chunk index within that file
    """
    chunks = []
    for path in Path(directory).rglob("*"):
        if path.suffix not in {".md", ".txt"}:
            continue
        text = path.read_text(encoding="utf-8", errors="ignore")
        start = 0
        idx = 0
        while start < len(text):
            end = start + chunk_size
            chunks.append({
                "text": text[start:end],
                "source": str(path),
                "chunk": idx,
            })
            start += chunk_size - overlap
            idx += 1
    print(f"Loaded {len(chunks)} chunks from '{directory}'")
    return chunks

## Step 2 — Generate embeddings

`all-MiniLM-L6-v2` is a lightweight but high-quality sentence embedding model that runs entirely locally — no API call required.
We L2-normalise the vectors so that inner-product search equals cosine similarity.

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def embed_texts(texts: list[str], model_name: str = EMBEDDING_MODEL) -> np.ndarray:
    """
    Encode *texts* with sentence-transformers and return L2-normalised float32 vectors.
    Shape: (len(texts), embedding_dim)
    """
    model = SentenceTransformer(model_name)
    vectors = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    # L2 normalise so inner product == cosine similarity
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return (vectors / norms).astype(np.float32)

## Step 3 — Build the FAISS index

`IndexFlatIP` performs exact inner-product search over all vectors — simple, reliable, and fast enough for up to a few hundred thousand chunks on a laptop.
For larger corpora, consider `IndexIVFFlat` or `IndexHNSWFlat` for approximate but much faster search.

In [ ]:
def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Build a FAISS inner-product index from *embeddings*.
    Vectors must already be L2-normalised.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index

## Step 4 — Retrieve relevant chunks

Given a query string, embed it the same way as the documents, then ask FAISS for the top-5 most similar chunks.

In [ ]:
def retrieve(
    query: str,
    index: faiss.IndexFlatIP,
    chunks: list[dict],
    top_k: int = 5,
    model_name: str = EMBEDDING_MODEL,
) -> list[dict]:
    """
    Embed *query*, search the FAISS *index*, and return the top-k matching chunks.
    Each returned dict has the original chunk fields plus a 'score' key.
    """
    q_vec = embed_texts([query], model_name=model_name)  # shape (1, dim)
    scores, indices = index.search(q_vec, top_k)
    results = []
    for score, i in zip(scores[0], indices[0]):
        if i == -1:          # FAISS uses -1 for "not found"
            continue
        result = dict(chunks[i])
        result["score"] = float(score)
        results.append(result)
    return results

## Step 5 — Answer with Claude

We format the retrieved chunks into a context block and ask Claude to answer **only** from that context.
Instructing Claude to cite the source file encourages transparent, traceable answers.

In [ ]:
CLAUDE_MODEL = "claude-opus-4-7"

SYSTEM_PROMPT = (
    "Answer based only on the provided context. "
    "Cite the source file for each piece of information you use. "
    "If the context does not contain enough information to answer, say so clearly."
)

def answer_with_claude(query: str, context_chunks: list[dict]) -> str:
    """
    Send *query* plus the retrieved *context_chunks* to Claude and return its answer.
    """
    context_text = "\n\n".join(
        f"[Source: {c['source']} | chunk {c['chunk']}]\n{c['text']}"
        for c in context_chunks
    )
    user_message = f"Context:\n{context_text}\n\nQuestion: {query}"

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

## Full pipeline example

Let's wire everything together with some sample documents created inline — no files on disk needed.

In [ ]:
import tempfile

# --- 1. Create temporary sample documents ---
SAMPLE_DOCS = {
    "getting_started.md": """
# Getting Started with Widget SDK

## Installation
Install the SDK with pip:
```
pip install widget-sdk
```

## Quick start
```python
from widget_sdk import Client
client = Client(api_key="your-key")
widget = client.create_widget(name="my-widget")
print(widget.id)
```

## Authentication
The SDK reads the WIDGET_API_KEY environment variable automatically.
You can also pass api_key directly to the Client constructor.
""",
    "faq.md": """
# Frequently Asked Questions

## How do I reset my API key?
Go to Settings > API Keys and click "Regenerate". Your old key will be immediately revoked.

## What is the rate limit?
Free tier: 100 requests per minute.
Pro tier: 10,000 requests per minute.
Enterprise: custom limits negotiated per contract.

## Which regions are supported?
US East, US West, EU West, and AP Southeast. Data residency guarantees are
available on Enterprise plans only.

## How do I contact support?
Email support@widget.io or open a ticket at help.widget.io.
""",
}

with tempfile.TemporaryDirectory() as doc_dir:
    # Write sample docs to disk
    for filename, content in SAMPLE_DOCS.items():
        Path(doc_dir, filename).write_text(content)

    # 2. Load and chunk
    chunks = load_and_chunk(doc_dir)

    # 3. Embed
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    # 4. Build index
    index = build_index(embeddings)

    # 5. Query
    query = "What is the rate limit for the Pro tier?"
    top_chunks = retrieve(query, index, chunks, top_k=5)

    print(f"\nTop retrieved chunk (score={top_chunks[0]['score']:.3f}):")
    print(top_chunks[0]["text"][:300])

    # 6. Answer
    answer = answer_with_claude(query, top_chunks)
    print("\n" + "="*60)
    print("QUESTION:", query)
    print("="*60)
    print(answer)

Expected output (your exact wording may vary):

```
Loaded 4 chunks from '/tmp/...' 
FAISS index built: 4 vectors, dim=384

============================================================
QUESTION: What is the rate limit for the Pro tier?
============================================================
Based on the FAQ document, the Pro tier rate limit is **10,000 requests per minute**.
(Source: faq.md | chunk 0)
```

## Next steps

- **Persist the index** — `faiss.write_index(index, "docs.index")` so you don't re-embed on every run.
- **Add PDF support** — use `pdfplumber` to extract text from PDFs before chunking.
- **Hybrid search** — combine dense vector search with BM25 keyword matching for better recall on exact terms.
- **Re-ranking** — use a cross-encoder to re-score the top-k before sending to Claude.
- **Full production app** — see [ask-my-docs](https://github.com/pintaste/ask-my-docs) for a complete FastAPI + FAISS service built on these same primitives.